# 2.3. Подготовка и организация данных

Подготавливаются доступные каталоги Steam и RAWG из `raw/`, сформированные этапом 2.2. Проверяются ключевые поля, создаётся осторожная таблица соответствий игр, подготовленные данные сохраняются в Parquet и DuckDB. Исходные файлы не изменяются.

In [36]:
from pathlib import Path
import json
import math
import unicodedata
import re

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import ijson

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "notebooks").is_dir()
        and (path / "README.md").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта. "
        "Запустите ноутбук внутри папки проекта."
    )

RAW = ROOT / "raw"
GAMES = ROOT / "processed" / "games"
WAREHOUSE = ROOT / "warehouse"

GAMES.mkdir(parents=True, exist_ok=True)
WAREHOUSE.mkdir(parents=True, exist_ok=True)

STEAM_SOURCE = RAW / "steam" / "fronkongames" / "games.json"
RAWG_SOURCE = RAW / "rawg" / "videogames_data.parquet"

for path in [STEAM_SOURCE, RAWG_SOURCE]:
    if not path.is_file():
        raise FileNotFoundError(f"Не найден файл из 2.2: {path}")

print("Корень проекта:", ROOT)
print("Steam:", STEAM_SOURCE.relative_to(ROOT))
print("RAWG:", RAWG_SOURCE.relative_to(ROOT))

Корень проекта: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Steam: raw\steam\fronkongames\games.json
RAWG: raw\rawg\videogames_data.parquet


## Подготовка и проверка качества

Для Steam сохраняются AppID, название, дата выпуска, цена, жанры, категории, положительные и отрицательные оценки, а также рекомендации. Для RAWG сохраняются ID, название и доступные характеристики игры.

Проверяются пропуски в ключевых полях и повторяющиеся идентификаторы и названия. Значения ID Steam и RAWG относятся к разным системам и не заменяют друг друга.

In [37]:
STEAM_OUTPUT = GAMES / "steam_games.parquet"

def to_text(value):
    if value is None:
        return None
    return str(value)

def to_float(value):
    try:
        number = pd.to_numeric(value, errors="coerce")
        return None if pd.isna(number) else float(number)
    except (TypeError, ValueError):
        return None

def to_int(value):
    number = to_float(value)
    if number is None or not math.isfinite(number):
        return None
    return int(number)

def to_json_text(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False)

steam_schema = pa.schema([
    ("AppID", pa.string()),
    ("name", pa.string()),
    ("release_date", pa.string()),
    ("price", pa.float64()),
    ("genres", pa.string()),
    ("categories", pa.string()),
    ("positive", pa.int64()),
    ("negative", pa.int64()),
    ("recommendations", pa.int64()),
])

steam_fields = [
    "name",
    "release_date",
    "price",
    "genres",
    "categories",
    "positive",
    "negative",
    "recommendations",
]

batch_size = 10_000
records = []
total_rows = 0
writer = None

try:
    with open(STEAM_SOURCE, "rb") as source:
        for app_id, game in ijson.kvitems(source, ""):
            records.append({
                "AppID": str(app_id),
                "name": to_text(game.get("name")),
                "release_date": to_text(game.get("release_date")),
                "price": to_float(game.get("price")),
                "genres": to_json_text(game.get("genres")),
                "categories": to_json_text(game.get("categories")),
                "positive": to_int(game.get("positive")),
                "negative": to_int(game.get("negative")),
                "recommendations": to_int(game.get("recommendations")),
            })

            if len(records) >= batch_size:
                table = pa.Table.from_pylist(records, schema=steam_schema)
                if writer is None:
                    writer = pq.ParquetWriter(
                        STEAM_OUTPUT,
                        steam_schema,
                        compression="zstd",
                    )
                writer.write_table(table)
                total_rows += len(records)
                records.clear()

    if records:
        table = pa.Table.from_pylist(records, schema=steam_schema)
        if writer is None:
            writer = pq.ParquetWriter(
                STEAM_OUTPUT,
                steam_schema,
                compression="zstd",
            )
        writer.write_table(table)
        total_rows += len(records)

finally:
    if writer is not None:
        writer.close()

if total_rows == 0:
    raise ValueError("В исходном Steam JSON не найдено записей.")

print("Steam: строк подготовлено =", total_rows)
print("Файл:", STEAM_OUTPUT.relative_to(ROOT))

Steam: строк подготовлено = 150466
Файл: processed\games\steam_games.parquet


In [39]:
RAWG_OUTPUT = GAMES / "rawg_games.parquet"

rawg_file = pq.ParquetFile(RAWG_SOURCE)
available_columns = set(rawg_file.schema_arrow.names)

rawg_columns = [
    "id",
    "name",
    "released",
    "rating",
    "rating_top",
    "ratings_count",
    "reviews_count",
    "metacritic",
    "playtime",
    "genres",
    "platforms",
    "developers",
    "publishers",
]

missing_required = {"id", "name"} - available_columns
if missing_required:
    raise ValueError(
        f"В RAWG Parquet отсутствуют обязательные поля: {missing_required}"
    )

rawg_columns = [
    column for column in rawg_columns
    if column in available_columns
]
rawg_schema = pa.schema([
    rawg_file.schema_arrow.field(column)
    for column in rawg_columns
])

writer = None
total_rows = 0

try:
    for batch in rawg_file.iter_batches(
        batch_size=50_000,
        columns=rawg_columns,
    ):
        if writer is None:
            writer = pq.ParquetWriter(
                RAWG_OUTPUT,
                rawg_schema,
                compression="zstd",
            )
        writer.write_batch(batch)
        total_rows += batch.num_rows

finally:
    if writer is not None:
        writer.close()

if total_rows == 0:
    raise ValueError("В исходном RAWG Parquet не найдено записей.")

print("RAWG: строк подготовлено =", total_rows)
print("Сохранённые поля:", rawg_columns)
print("Файл:", RAWG_OUTPUT.relative_to(ROOT))

RAWG: строк подготовлено = 899585
Сохранённые поля: ['id', 'name', 'released', 'rating', 'rating_top', 'ratings_count', 'reviews_count', 'metacritic', 'playtime', 'genres', 'platforms', 'developers', 'publishers']
Файл: processed\games\rawg_games.parquet


## Краткий профиль данных

Сводка показывает размер подготовленных таблиц, пропуски и дубликаты в ключевых полях. Повторяющиеся названия учитываются отдельно, поскольку одно название может относиться к нескольким записям.

In [40]:
steam_keys = pd.read_parquet(
    STEAM_OUTPUT,
    columns=["AppID", "name"],
)
rawg_keys = pd.read_parquet(
    RAWG_OUTPUT,
    columns=["id", "name"],
)

quality = pd.DataFrame([
    {
        "source": "Steam",
        "rows": len(steam_keys),
        "missing_id": int(steam_keys["AppID"].isna().sum()),
        "missing_name": int(steam_keys["name"].isna().sum()),
        "duplicate_id": int(steam_keys["AppID"].duplicated().sum()),
        "names_used_more_than_once": int(
            steam_keys["name"].dropna().value_counts().gt(1).sum()
        ),
    },
    {
        "source": "RAWG",
        "rows": len(rawg_keys),
        "missing_id": int(rawg_keys["id"].isna().sum()),
        "missing_name": int(rawg_keys["name"].isna().sum()),
        "duplicate_id": int(rawg_keys["id"].duplicated().sum()),
        "names_used_more_than_once": int(
            rawg_keys["name"].dropna().value_counts().gt(1).sum()
        ),
    },
])

display(quality)

,source,rows,missing_id,missing_name,duplicate_id,names_used_more_than_once
0,Steam,150466,0,0,0,1296
1,RAWG,899585,0,0,0,15


## Сопоставление Steam и RAWG

Названия нормализуются только для поиска совпадений. В итоговую таблицу входят совпадения, для которых название встречается ровно один раз в каждом каталоге. Это кандидаты на соответствие, а не подтверждённые связи; исходные ID и названия сохраняются раздельно.

In [41]:
def normalize_game_name(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value)).casefold()
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"[_\s]+", " ", value)

    return value.strip() or None

steam_games = pd.read_parquet(
    STEAM_OUTPUT,
    columns=["AppID", "name"],
)
rawg_games = pd.read_parquet(
    RAWG_OUTPUT,
    columns=["id", "name"],
)

steam_games["normalized_name"] = (
    steam_games["name"].map(normalize_game_name)
)
rawg_games["normalized_name"] = (
    rawg_games["name"].map(normalize_game_name)
)

steam_counts = steam_games["normalized_name"].value_counts()
rawg_counts = rawg_games["normalized_name"].value_counts()

steam_unique = steam_games[
    steam_games["normalized_name"].map(steam_counts).eq(1)
].dropna(subset=["normalized_name"])

rawg_unique = rawg_games[
    rawg_games["normalized_name"].map(rawg_counts).eq(1)
].dropna(subset=["normalized_name"])

matches = steam_unique.merge(
    rawg_unique,
    on="normalized_name",
    how="inner",
    suffixes=("_steam", "_rawg"),
    validate="one_to_one",
)

matches = matches.rename(columns={
    "AppID": "steam_app_id",
    "id": "rawg_id",
    "name_steam": "steam_name",
    "name_rawg": "rawg_name",
})

matches["match_method"] = "exact_normalized_name"
matches["match_status"] = "unique_name_match_unverified"

matches = matches.sort_values(
    ["steam_app_id", "rawg_id"]
).reset_index(drop=True)

MATCHES_OUTPUT = GAMES / "steam_rawg_matches.parquet"
matches.to_parquet(
    MATCHES_OUTPUT,
    index=False,
    compression="zstd",
)

print("Однозначных совпадений-кандидатов:", len(matches))
print("Файл:", MATCHES_OUTPUT.relative_to(ROOT))
display(matches.head())

Однозначных совпадений-кандидатов: 102889
Файл: processed\games\steam_rawg_matches.parquet


,steam_app_id,steam_name,normalized_name,rawg_id,rawg_name,match_method,match_status
0,1000000,ASCENXION,ascenxion,306066,ASCENXION,exact_normalized_name,unique_name_match_unverified
1,1000010,Crown Trick,crown trick,322240,Crown Trick,exact_normalized_name,unique_name_match_unverified
2,1000030,"Cook, Serve, Delicious! 3?!",cook serve delicious 3,362860,"Cook, Serve, Delicious! 3?!",exact_normalized_name,unique_name_match_unverified
3,1000050,我的一个多重人格朋友,我的一个多重人格朋友,996472,我的一个多重人格朋友,exact_normalized_name,unique_name_match_unverified
4,1000080,Zengeon,zengeon,537747,Zengeon,exact_normalized_name,unique_name_match_unverified


## Аналитическое хранилище

Подготовленные Parquet-таблицы загружаются в DuckDB. В базе хранятся отдельные таблицы Steam и RAWG, а также таблица кандидатов на сопоставление. Исходные ID платформ не объединяются в один идентификатор.

In [42]:
DATABASE = WAREHOUSE / "games.duckdb"

parquet_tables = {
    "steam_games": STEAM_OUTPUT,
    "rawg_games": RAWG_OUTPUT,
    "steam_rawg_matches": MATCHES_OUTPUT,
}

connection = duckdb.connect(str(DATABASE))

try:
    for table_name, parquet_path in parquet_tables.items():
        # Путь берётся из Path, одинарные кавычки экранируются для SQL.
        sql_path = parquet_path.resolve().as_posix().replace("'", "''")

        connection.execute(
            f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT * FROM read_parquet('{sql_path}')
            """
        )

    table_summary = []
    for table_name in parquet_tables:
        row_count = connection.execute(
            f"SELECT COUNT(*) FROM {table_name}"
        ).fetchone()[0]

        table_summary.append({
            "table": table_name,
            "rows": row_count,
        })

finally:
    connection.close()

print("База данных:", DATABASE.relative_to(ROOT))
display(pd.DataFrame(table_summary))

База данных: warehouse\games.duckdb


,table,rows
0,steam_games,150466
1,rawg_games,899585
2,steam_rawg_matches,102889


## Итог

Каталоги Steam и RAWG подготовлены в Parquet. Качество ключевых полей кратко проверено. Потенциальные связи игр сформированы по уникальным нормализованным названиям и помечены как непроверенные. Таблицы сохранены в DuckDB для последующего анализа.

Steam Reviews и Epic Games в этом ноутбуке не обрабатываются, поскольку соответствующие исходные файлы не доступны в используемом наборе данных.